# Homework 4 — solution by (your_name)

Read the homework PDF first: it has the rules and the suggested tables. The questions are quoted in this notebook, in
blockquotes (with a bar on their left), so that your answers stand apart. Copy this notebook to
`3dasm_course/Assignments/your_Assignments/HW4/your_Homework4.ipynb` and work there.

**How this notebook works** (as in the lectures):

1. the cell `%%writefile CLAUDE.md` writes the instructions file that every agent of this homework reads: **you** write it;
2. each prompt is a `PROMPT_...` cell, run with `run_agent`: **you** write the prompts. Every agent works in its own
   task folder inside `HW4/` (e.g. `HW4/ex4_4/`), so from an agent's point of view the data folder
   `3dasm_course/Assignments/your_data/` is `../../../your_data/`;
3. after each prompt, a cell that we provide loads the agent's `results.json`. The keys it needs are written just above
   each prompt cell: your prompt must make the agent store exactly those. If one is missing, the cell tells you which.

In [ ]:
# Provided by the course: the helper that runs a coding agent from this notebook (see agent_helper.py)
import sys; sys.path.insert(0, "../../..")          # the folder 3dasm_course, where agent_helper.py is
from agent_helper import set_model, run_agent, results, figures, show, save_results, task_folder
import numpy as np, pandas as pd

set_model("sonnet")        # "sonnet" or "haiku": the TA runs your notebook again with this same model
import sklearn; print("numpy", np.__version__, "| scikit-learn", sklearn.__version__)   # some results depend on them

## Your instructions file for the agents

Write here everything that is the same for all prompts: the environment, the packages, where files go, how randomness
must be handled, and what the agent must report. The lectures have an example.

In [ ]:
%%writefile CLAUDE.md
# YOUR CLAUDE.md HERE

## Exercise 1 (pen and paper)

> **1.1.** Explain from a Bayesian perspective the Linear Least Squares model and why do we use it to fit data with polynomial basis functions.

*YOUR ANSWER HERE*

> **1.2.** What is a point estimate?

*YOUR ANSWER HERE*

> **1.3.** Explain what are training, validation and testing sets.

*YOUR ANSWER HERE*

> **1.4.** Define and explain the $R^2$ metric.

*YOUR ANSWER HERE*

> **1.5.** Define and explain what is the $\text{MSE}$ metric.

*YOUR ANSWER HERE*

> **1.6.** Explain what is $k$-fold cross-validation.

*YOUR ANSWER HERE*

## Exercise 2 (pen and paper)

> **2.1.** Consider a two-dimensional input $\mathbf{x} = [x_1, x_2]^T$ and the quadratic basis functions vector in the order used by scikit-learn: $$\boldsymbol{\phi}(\mathbf{x}) = [1,\ x_1,\ x_2,\ x_1^2,\ x_1 x_2,\ x_2^2]^T.$$ Write the matrix of basis functions $\boldsymbol{\Phi}$ for the three training points $\mathbf{x}^{(1)} = [1, 2]^T$, $\mathbf{x}^{(2)} = [0, -1]^T$ and $\mathbf{x}^{(3)} = [2, 0]^T$. Give the dimensions of $\boldsymbol{\Phi}$, $\mathbf{w}$ and $\mathbf{y}$.

*YOUR ANSWER HERE*

> **2.2.** Is $\boldsymbol{\Phi}^T\boldsymbol{\Phi}$ invertible for the example of 2.1? Justify your answer using the rank of $\boldsymbol{\Phi}$. More generally, what must be true about the number of training points $N$ and the number of basis functions $M$ for $\boldsymbol{\Phi}^T\boldsymbol{\Phi}$ to be invertible?

*YOUR ANSWER HERE*

> **2.3.** The maximum likelihood estimate of the noise variance is $\hat{\sigma}^2 = \text{RSS}/N$. Explain in your own words why it underestimates $\sigma^2$, and write the unbiased alternative. When does the difference matter?

*YOUR ANSWER HERE*

> **2.4.** Write the plug-in posterior predictive distribution for a new input $x^*$. Which source of uncertainty does it ignore?

*YOUR ANSWER HERE*

## Exercise 3 (pen and paper)

Consider a linear regression model with two outputs, $\mathbf{y} = [y_1, y_2]^T$: $$p(\mathbf{y}|\mathbf{x}, \mathbf{W}, \boldsymbol{\Sigma}) = \mathcal{N}\left(\mathbf{y}\,|\,\mathbf{W}^T\boldsymbol{\phi}(\mathbf{x}),\ \boldsymbol{\Sigma}\right),$$ where $\boldsymbol{\phi}(\mathbf{x})$ has $M$ basis functions, $\mathbf{W}$ has one column per output, and $\boldsymbol{\Sigma}$ is a $2\times 2$ covariance matrix, possibly with non-zero off-diagonal terms. The $N$ training outputs are collected in the matrix $\mathbf{Y}$, one row per point.

> **3.1.** Give the dimensions of $\mathbf{W}$, $\mathbf{Y}$ and $\boldsymbol{\Phi}$. Is $\boldsymbol{\Phi}$ different from the single-output case?

*YOUR ANSWER HERE*

> **3.2.** Write the negative log likelihood and show that its minimum with respect to $\mathbf{W}$ is $\hat{\mathbf{W}} = (\boldsymbol{\Phi}^T\boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^T\mathbf{Y}$, whatever $\boldsymbol{\Sigma}$ is. What does this mean for the columns of $\hat{\mathbf{W}}$? (Hint: write the condition that the gradient with respect to $\mathbf{W}$ is zero, and show that $\boldsymbol{\Sigma}^{-1}$ disappears when you multiply it on the right by $\boldsymbol{\Sigma}$.)

*YOUR ANSWER HERE*

> **3.3.** Show that the maximum likelihood estimate of the covariance is $\hat{\boldsymbol{\Sigma}} = \frac{1}{N}(\mathbf{Y}-\boldsymbol{\Phi}\hat{\mathbf{W}})^T(\mathbf{Y}-\boldsymbol{\Phi}\hat{\mathbf{W}})$.

*YOUR ANSWER HERE*

## Exercise 4 (with the agent)

Consider the function to be learned as $f(x)=x \sin(x)$ within the domain $x\in [0,10]$.

> **4.1.** Create a dataset without noise from $f(x)$ as follows (Homeworks 5 and 6 will use this exact file, so follow these instructions precisely):
>
> -   Sample 40 values of $x$ uniformly at random in $[0,10]$, in a single draw from NumPy’s default random number generator with seed 123 (`np.random.default_rng`). Note that “seed 123” alone is not enough: the older `np.random.seed(123)` gives different points.
>
> -   Sort the points in ascending order of $x$, and save them as a pandas dataframe with two columns, `x` and `y`, in a file called “HW4_noiseless_dataset.csv” (without the index) in the existing folder “your_data”.
>
> -   Split the sorted dataset into a training set with 80% of the points and a testing set with the remaining 20%, with the “train_test_split” function of scikit-learn and its “random_state” set to 123.
>
> -   **Important**: use the same training and testing sets for all the models of this homework.

Your prompt must make the agent store in `results.json`: `x_test` (the testing values of $x$, in the order returned by the split). It must also save the dataset file.

In [ ]:
TASK_4_1 = "ex4_1"          # the task name (the cells below use it)
PROMPT_4_1 = """YOUR PROMPT HERE"""
run_agent(TASK_4_1, PROMPT_4_1)

In [ ]:
# Provided: the results of task ex4_1
print(pd.read_csv("../../your_data/HW4_noiseless_dataset.csv").head())
print("testing x:", np.round(sorted(results(TASK_4_1, ["x_test"])["x_test"]), 2))

> **4.2.** Before comparing models, check that the computer solves the least squares problem correctly. With the entire training set (32 points), fit polynomials of every degree from 1 to 20 twice: once on the raw $x$ (scikit-learn’s `PolynomialFeatures` and `LinearRegression`, with default arguments), and once with $x$ first rescaled to the interval $[-1, 1]$ by scikit-learn’s `MinMaxScaler`, fitted on these training points. Plot the training $\text{MSE}$ of both against the degree, with a logarithmic vertical axis. A polynomial of degree $d+1$ can reproduce any polynomial of degree $d$ (set its last weight to zero): what does this imply for the training $\text{MSE}$ as the degree grows? Which of the two curves obeys it? (Your raw curve may differ from a classmate’s, depending on the version of scikit-learn: 5.3 explains why.) **From now on, rescale $x$ in this way in every model of this exercise.**

Your prompt must make the agent store in `results.json`: `train_mse_raw` and `train_mse_scaled`, each a list of 20 values (degrees 1 to 20).

In [ ]:
TASK_4_2 = "ex4_2"          # the task name (the cells below use it)
PROMPT_4_2 = """YOUR PROMPT HERE"""
run_agent(TASK_4_2, PROMPT_4_2)

In [ ]:
# Provided: the results of task ex4_2
r = results(TASK_4_2, ["train_mse_raw", "train_mse_scaled"])
print(pd.DataFrame({"raw x": r["train_mse_raw"], "x in [-1, 1]": r["train_mse_scaled"]},
                   index=[f"deg {d}" for d in range(1, 21)]).to_string(float_format="%.2e"))
show(*figures(TASK_4_2), width=600)

*YOUR ANSWER HERE*

> **4.3.** **Before running anything else**, write down your predictions: when the models of the next question are trained with $N = 6$, $11$, $21$ and $32$ points, which polynomial degree do you expect to predict the testing points best, and what do you expect each model to do outside the domain $x\in [0,10]$? (You will not lose points for wrong predictions. You will lose points for not writing them.)

*YOUR ANSWER HERE*

> **4.4.** Train Linear Least Squares models, with $x$ rescaled as in 4.2, but with the `MinMaxScaler` fitted on the training points of each model only, with polynomials of degree 1, 3, 5, 10 and 20, using four different training set sizes: the first 6, 11, 21 and 32 points of the training set of 4.1, in the order in which “train_test_split” returns them. For every model, calculate the $\text{MSE}$ on its training points, and the $R^2$ and $\text{MSE}$ on the testing set. Present them as suggested in Table 1 (in the PDF), and show 4 figures (one per training set size) with the training points, the testing points, $f(x)$ and the polynomial approximations, for $x\in[-2, 12]$.

Your prompt must make the agent store in `results.json`: `train_mse`, `test_mse` and `test_r2`, each a 4×5 list (rows: $N$ = 6, 11, 21, 32; columns: degree 1, 3, 5, 10, 20).

In [ ]:
TASK_4_4 = "ex4_4"          # the task name (the cells below use it)
PROMPT_4_4 = """YOUR PROMPT HERE"""
run_agent(TASK_4_4, PROMPT_4_4)

In [ ]:
# Provided: the results of task ex4_4
def table(r, key):
    return pd.DataFrame(r[key], index=[f"N={n}" for n in (6, 11, 21, 32)], columns=[f"deg {d}" for d in (1, 3, 5, 10, 20)])
r = results(TASK_4_4, ["train_mse", "test_mse", "test_r2"])
for key in ("test_r2", "test_mse", "train_mse"):
    print(key, "\n", table(r, key).to_string(float_format="%.2e"), "\n")
show(*figures(TASK_4_4), width=600)

> **4.5.** Now consider the **entire training set** (32 points) and use 4-fold cross-validation, with the points shuffled before they are divided into folds (with seed 123), to calculate the **mean** and **standard deviation** of the $R^2$ and $\text{MSE}$ metrics for the same five models as in 4.4 (the scaler is then fitted on the training folds only; the standard deviation is that of the 4 values, as computed by NumPy). Present the results as suggested in Table 2 (in the PDF). **Explain the obtained mean and standard deviation**. Note that each fold has as many points as the testing set.

Your prompt must make the agent store in `results.json`: `cv_mse_mean`, `cv_mse_std`, `cv_r2_mean` and `cv_r2_std`, each a list with one value per degree (1, 3, 5, 10, 20).

In [ ]:
TASK_4_5 = "ex4_5"          # the task name (the cells below use it)
PROMPT_4_5 = """YOUR PROMPT HERE"""
run_agent(TASK_4_5, PROMPT_4_5)

In [ ]:
# Provided: the results of task ex4_5
keys = ["cv_mse_mean", "cv_mse_std", "cv_r2_mean", "cv_r2_std"]
r = results(TASK_4_5, keys)
print(pd.DataFrame({k: r[k] for k in keys}, index=[f"deg {d}" for d in (1, 3, 5, 10, 20)]).to_string(float_format="%.3f"))

*YOUR ANSWER HERE*

> **4.6.** **What can you conclude from these results?** Compare them with your predictions of 4.3, using the terms underfitting, overfitting, interpolation and extrapolation. In particular:
>
> -   For which training set sizes and degrees is the training $\text{MSE}$ (almost) zero, and why?
>
> -   Where are the testing errors large, and why? (Hint: for each training set size, compare where the testing points are with where the training points are.)
>
> -   Is the testing $R^2$ a good guide here? Look at where the testing points are, and compare with the cross-validation of 4.5..
>
> -   Compare with the noisy example seen in class: where does overfitting come from in each case?

*YOUR ANSWER HERE*

## Exercise 5 (with the agent)

> **5.1.** For the model of degree 3 on the raw $x$ (without rescaling), trained with the first 11 training points of 4.4, report the weights obtained by scikit-learn, and the weights $\hat{\mathbf{w}} = (\boldsymbol{\Phi}^T\boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^T\mathbf{y}$ computed explicitly with NumPy, with the inverse of the matrix (not with a least-squares solver). If computing the inverse fails, your code must record the error and continue; if it does not fail, that does not mean that its result is right. Do the two sets of weights agree? Quote the lines of the agent’s code where $\boldsymbol{\Phi}$ is built, and explain where the scikit-learn model stores the weight $w_0$.

> **5.2.** Repeat 5.1 for the model of degree 20 trained with the first 6 training points. Do the two sets of weights agree? Plot both models on the same figure. In exact arithmetic, how many polynomials of degree 20 pass through 6 points (use your answer to 2.2)? Do the two models pass through the 6 training points? Report the smallest and the largest entries of $\boldsymbol{\Phi}$, and explain why the computer struggles with this problem even though the math is simple. (Hint: a computer keeps about 16 significant digits; compare that with the ratio of the largest to the smallest entry of $\boldsymbol{\Phi}$.)

Your prompt must make the agent store in `results.json`: `N11_deg3` and `N6_deg20`, each a dictionary with `w_sk` and `w_inv` (the $M+1$ weights of scikit-learn's model and of the inverse, in the order of the columns of $\boldsymbol{\Phi}$: $w_0$ first; `null` if the inverse failed), `inv_error` (the error message, or `null`), `max_abs_diff` (the largest $|w_{sk} - w_{inv}|$), `train_max_abs_err_sk` (the largest $|\hat y - y|$ of scikit-learn's model on its training points), and `Phi_min_abs` and `Phi_max_abs` (the smallest and largest absolute entries of $\boldsymbol{\Phi}$).

In [ ]:
TASK_5 = "ex5"              # the task name (the cells below use it)
PROMPT_5 = """YOUR PROMPT HERE"""
run_agent(TASK_5, PROMPT_5)

In [ ]:
# Provided: the results of task ex5
for case, v in results(TASK_5, ["N11_deg3", "N6_deg20"]).items():
    print(case, {k: val for k, val in v.items() if not k.startswith("w_")})
    for k in ("w_sk", "w_inv"):
        print(f"   {k:5} =", np.round(v[k], 6).tolist() if v[k] is not None else None)
show(*figures(TASK_5), width=600)

**5.1.** *YOUR ANSWER HERE*

**5.2.** *YOUR ANSWER HERE*

> **5.3.** Which of the infinitely many solutions does scikit-learn return in that case? Which directions does it treat as zero, and does this explain the raw curve of 4.2? (Hint: read the documentation of `LinearRegression`, including its parameter `tol`, and of the function it uses to solve the problem.)

*YOUR ANSWER HERE*

## Exercise 6 (with the agent)

> **6.1.** Create a dataset with two noisy outputs: with NumPy’s default random number generator and seed 123, first draw 40 values of $x$ uniformly at random in $[0,10]$, and then 40 noise vectors $[\epsilon_1, \epsilon_2]^T$ with the generator’s `multivariate_normal`, with zero mean and covariance $\boldsymbol{\Sigma} = \begin{bmatrix}1 & 0.8\\ 0.8 & 1\end{bmatrix}$. The outputs are $y_1 = x\sin(x) + \epsilon_1$ and $y_2 = x\cos(x) + \epsilon_2$. Split the dataset into training (80%) and testing (20%) sets with “train_test_split” and its “random_state” set to 123. With polynomial basis functions of degree 5 and the training set, verify that three computations give the same weights: one scikit-learn model fitted to both outputs at once, two scikit-learn models fitted to one output each, and the expression of 3.2. Report $\hat{\boldsymbol{\Sigma}}$ (the maximum likelihood estimate of 3.2, from the training residuals), and compare it with the true $\boldsymbol{\Sigma}$.

Your prompt must make the agent store in `results.json`: `max_abs_diff_joint` and `max_abs_diff_separate` (the largest difference between the weights of the expression of 3.2 and those of the joint fit, and of the two separate fits) and `Sigma_hat` (a 2×2 list).

In [ ]:
TASK_6 = "ex6"              # the task name (the cells below use it)
PROMPT_6 = """YOUR PROMPT HERE"""
run_agent(TASK_6, PROMPT_6)

In [ ]:
# Provided: the results of task ex6
r = results(TASK_6, ["max_abs_diff_joint", "max_abs_diff_separate", "Sigma_hat"])
S = np.array(r["Sigma_hat"])
print({k: r[k] for k in ("max_abs_diff_joint", "max_abs_diff_separate")})
print("Sigma_hat =", np.round(S, 3).tolist(), "  correlation: %.3f" % (S[0, 1] / np.sqrt(S[0, 0] * S[1, 1])))
show(*figures(TASK_6), width=700)

*YOUR ANSWER HERE*

> **6.2.** If $\boldsymbol{\Sigma}$ does not change the weights, what does knowing it give you? Give an example of a decision that depends on both outputs at the same time, for which the off-diagonal term of $\boldsymbol{\Sigma}$ matters.

*YOUR ANSWER HERE*